# ControlNet & conditioning — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## residual spatial conditioning
Latent diffusion (10.14) supplies the frozen generator, classifier-free guidance (10.13) supplies prompt steering, and residual networks explain how an added branch can influence layers. This enables pose, edge, depth, and layout control for generative systems.

In [ ]:
# 1) residual conditioning adds a control feature
base=np.array([.4,.1]); cond=np.array([.3,-.2]); out=base+cond
plt.figure(figsize=(4,4)); plt.quiver([0,0],[0,0],[base[0],out[0]],[base[1],out[1]],angles='xy',scale_units='xy',scale=1); plt.title('base feature plus control residual')
assert np.allclose(out,[.7,-.1])

In [ ]:
# 2) zero-initialized branch starts with no change
scales=np.linspace(0,1,50); change=[np.linalg.norm(base+s*cond-base) for s in scales]
plt.figure(figsize=(5,3)); plt.plot(scales,change); plt.xlabel('branch scale'); plt.title('control grows from zero')
assert change[0]==0 and change[-1]>0

In [ ]:
# 3) edge-like condition supplies spatial structure
img=np.zeros((20,20)); img[5:15,5]=1; img[5:15,15]=1; img[5,5:16]=1; img[15,5:16]=1
plt.figure(figsize=(4,4)); plt.imshow(img,cmap='gray'); plt.title('toy edge condition')
assert img.sum()==40

In [ ]:
# 4) strong residual can override the base feature
s=np.linspace(0,3,50); norm=[np.linalg.norm(base+ss*cond) for ss in s]
plt.figure(figsize=(5,3)); plt.plot(s,norm); plt.title('control strength changes feature magnitude')
assert norm[-1]>norm[0]

In [ ]:
# 5) two conditions combine as separate residuals
pose=np.array([.2,.0]); depth=np.array([.0,-.3]); both=base+pose+depth
plt.figure(figsize=(4,4)); plt.quiver([0,0,0],[0,0,0],[pose[0],depth[0],both[0]],[pose[1],depth[1],both[1]],angles='xy',scale_units='xy',scale=1); plt.title('multiple conditioning residuals')
assert np.allclose(both,[.6,-.2])